In [7]:
# import required libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import contextlib

In [14]:
# Load data
SCHOOLS_CSV = 'kenya_primary_schools.csv'             
POP_SUBCOUNTY = 'kenya-population-by-sub-county.csv' 
POP_AGE_SUBCOUNTY = 'distribution-of-population-age-3-years-and-above-by-county.csv'
POP_DENSITY = 'kenya-populationland-area-population-density.csv'
DISABILITY = 'distribution-of-population-aged-5-years-and-above-by-disability.csv'  

In [15]:

# 1. Load school data
schools = pd.read_csv(SCHOOLS_CSV)


FileNotFoundError: [Errno 2] No such file or directory: 'kenya_primary_schools.csv'

In [10]:
# quick inspect
print(schools.columns.tolist())


NameError: name 'schools' is not defined

In [16]:
# create geometry
schools = schools.dropna(subset=['Latitude','Longitude'])
schools['Latitude'] = pd.to_numeric(schools['Latitude'], errors='coerce')
schools['Longitude'] = pd.to_numeric(schools['Longitude'], errors='coerce')
schools = schools.dropna(subset=['Latitude','Longitude'])
g_sch = gpd.GeoDataFrame(schools, geometry=gpd.points_from_xy(schools.Longitude, schools.Latitude), crs='EPSG:4326')


NameError: name 'schools' is not defined

In [ ]:
# standardize admin text for join keys
def clean_text(s): 
    try: return str(s).strip().upper()
    except: return s
g_sch['DISTRICT_CLEAN'] = g_sch['District'].map(clean_text)
g_sch['CONSTITUENCY_CLEAN'] = g_sch['Costituenc'].map(clean_text)


In [ ]:
oad population datasets
pop_sub = pd.read_csv(POP_SUBCOUNTY)
pop_age = pd.read_csv(POP_AGE_SUBCOUNTY)  # contains age groups -> filter ages 6-13
pop_density = pd.read_csv(POP_DENSITY)
disab = pd.read_csv(DISABILITY, encoding='latin1')  # encoding may vary

# example: compute primary-age population by sub-county
# Adjust column names as per your file, below assumes 'SubCounty','Age','Total'
primary_ages = pop_age[pop_age['Age'].isin([6,7,8,9,10,11,12,13])]
primary_by_sc = primary_ages.groupby(['SubCounty'])['Total'].sum().reset_index().rename(columns={'Total':'primary_pop'})
primary_by_sc['SubCounty_clean'] = primary_by_sc['SubCounty'].map(clean_text)

# -------------------
# 3. Spatial aggregation of supply to sub-county
# -------------------
# If you have a sub-county shapefile, use spatial join. Otherwise aggregate by text key.
# Option A: text aggregation by district/constituency
supply_agg = g_sch.groupby('CONSTITUENCY_CLEAN').agg(
    schools_count=('FID','count'),
    total_classrooms=('No_Classrm', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    total_teachers=('NoTeaching', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    total_enrol=('TotalEnrol', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    boys=('TotalBoys', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    girls=('TotalGirls', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    girls_toilets=('GirlsToilet', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    boys_toilets=('BoysToilet', lambda x: pd.to_numeric(x, errors='coerce').sum())
).reset_index().rename(columns={'CONSTITUENCY_CLEAN':'constituency'})

# -------------------
# 4. Merge supply + demand
# -------------------
# ensure matching keys: make a POP key that matches your supply_agg key
primary_by_sc['constituency'] = primary_by_sc['SubCounty_clean']  # rename if needed
df = supply_agg.merge(primary_by_sc[['constituency','primary_pop']], on='constituency', how='left').fillna(0)

# -------------------
# 5. Derived metrics
# -------------------
df['total_classrooms'] = df['total_classrooms'].replace(0, np.nan)
df['total_teachers'] = df['total_teachers'].replace(0, np.nan)
df['pupil_per_classroom'] = df['total_enrol'] / df['total_classrooms']
df['pupil_per_teacher'] = df['total_enrol'] / df['total_teachers']
# capacity estimate fallback
df['capacity_estimate'] = df['total_classrooms'] * 40
df['capacity_shortfall'] = df['primary_pop'] - df['capacity_estimate']
df['girls_toilet_per_100girls'] = (df['girls_toilets'] / df['girls']) * 100

# -------------------
# 6. Vulnerability index
# -------------------
scaler = MinMaxScaler()
# create scaled columns, fillna=0 for scaling
df[['shortfall_s']] = scaler.fit_transform(df[['capacity_shortfall']].fillna(0).values)
df[['pcr_s']] = scaler.fit_transform(df[['pupil_per_classroom']].fillna(0).values)
df[['ptr_s']] = scaler.fit_transform(df[['pupil_per_teacher']].fillna(0).values)
# Example weights
df['vulnerability'] = 0.4*df['shortfall_s'] + 0.35*df['pcr_s'] + 0.25*df['ptr_s']
df = df.sort_values('vulnerability', ascending=False)

# -------------------
# 7. Quick export for slides
# -------------------
df[['constituency','primary_pop','total_enrol','capacity_estimate','capacity_shortfall','pupil_per_classroom','pupil_per_teacher','vulnerability']].head(10).to_csv('outputs/top_constituencies.csv', index=False)

# -------------------
# 8. Simple visuals
# -------------------
plt.figure(figsize=(10,6))
df.head(10).plot(kind='bar', x='constituency', y='vulnerability', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Constituencies by Vulnerability Index')
plt.tight_layout()
plt.savefig('outputs/top10_vulnerability.png', dpi=200)

# Map: if you have constituency shapefile, merge and plot a choropleth of vulnerability
# wards_shp = gpd.read_file('nairobi_constituencies.shp').to_crs('EPSG:4326')
# wards_shp['NAME_CLEAN']=wards_shp['NAME'].map(clean_text)
# mapdf = wards_shp.merge(df, left_on='NAME_CLEAN', right_on='constituency', how='left')
# mapdf.plot(column='vulnerability', cmap='OrRd', legend=True, figsize=(10,8))
# plt.savefig('outputs/vulnerability_map.png', dpi=200)
